# 🧩 Gemma 4 (E4B) Fine-Tuning — Metinden Yapılandırılmış JSON Çıkarma
### Unsloth + QLoRA ile Kaggle / Colab üzerinde, ücretsiz T4 GPU

**Antalya — Büyük Dil Modelleri (LLMs) Tabanlı Uygulama Geliştirme Eğitimi**

---

## Bu notebook'ta ne yapacağız, niçin?

Serbest, dağınık bir metni (fatura, tıbbi kayıt, sipariş, haber…) **verilen JSON şemasına birebir uyan** temiz bir JSON'a çeviren bir model eğiteceğiz.

**Niçin bu görev?** Gerçek üretimde en çok işe yarayan LLM görevlerinden biri: belgeden veri çıkarıp veritabanına / otomasyona besleme (*schema-guided structured extraction*). Çıktı geçerli JSON olduğunda `json.loads()` ile doğrudan koda akar.

**Niçin fine-tune, niçin sadece prompt değil?**
> **Analoji:** Prompt vermek, birine *her seferinde* "lütfen şu formatta yaz" diye not bırakmaktır. Fine-tune, o kişiyi *eğitip* o formatı kas hafızasına yerleştirmektir — artık söylemenize gerek kalmaz.

- Küçük bir modele prompt ile JSON istediğinizde çoğu zaman "İşte JSON'unuz:" gibi gevezelik ekler, şemadan sapar, alan uydurur.
- Fine-tune ile modeli **sadece geçerli JSON** üretmeye şartlarız → çıktı doğrudan parse edilir.
- Küçük model (E4B) + QLoRA → tek **ücretsiz** T4 GPU'da eğitilir, laptopta/Ollama'da çalışır. API ücreti yok, veri dışarı çıkmaz (KVKK/gizlilik).

**Yığın:**
| Bileşen | Seçim | Niçin |
|---|---|---|
| Model | `unsloth/gemma-4-E4B-it` | Google açık ağırlık; **E4B = efektif ~4B** aktif parametre, T4'e sığar |
| Yöntem | QLoRA (4-bit) | Tam fine-tune yerine ~%0.5 parametre → 16GB VRAM yeter |
| Kütüphane | Unsloth | Hugging Face Transformers üzerine kurulu; ~1.6× hız, %60 daha az VRAM |
| Veri | `paraloq/json_data_extraction` | 484 örnek, 8 alan; (metin+şema→JSON), text-only |

> ⚙️ **Kaggle:** Sağ panel → *Accelerator* = **GPU T4 x2**, *Internet* = **On**.
> Bu notebook, Murat Karakaya Akademi'nin "Gemma 4 İnce Ayar" anlatımındaki akışı, hazır bir veri setiyle uçtan uca ölçülebilir bir uygulamaya dönüştürür.


## 📚 Önce iki kavram (5 dakikalık teori)

**1) Base (temel) model vs Instruct (-it) model**
- **Base model** ham bir *sonraki-kelime tahmincisidir*. "Türkiye'nin başkenti neresidir?" dediğinizde derdi cevap vermek değil, cümleyi *tamamlamaktır*. Kendi verinizle sıfırdan hizalamak (alignment) için uygundur.
- **Instruct model** (`-it`) zaten komut anlayıp cevap verecek şekilde hizalanmıştır. Biz bunu seçiyoruz — çünkü sadece **davranışı inceltmek** (JSON disiplini) istiyoruz, sıfırdan konuşmayı öğretmek değil.

**2) Quantization (nicemleme) — niçin 4-bit?**
- Modelin 16-bit hali diskte ~10 GB'tan fazla yer kaplar; T4'e sığmaz.
- **4-bit** sıkıştırılmış hali ~5–7 GB'a iner → ücretsiz GPU'ya sığar. Doğrulukta ihmal edilebilir kayıpla, belleği ~2.5× küçültürüz.

**3) QLoRA / PEFT — niçin tam fine-tune yapmıyoruz?**
- Tam fine-tune (8B ağırlığın hepsini eğitmek) ciddi GPU gücü ister; ücretsiz ortamda mümkün değil.
- **PEFT (Parameter-Efficient Fine-Tuning)** ailesinden **LoRA**: ağırlıkları dondurur, araya küçük eğitilebilir matrisler koyar. **QLoRA** = LoRA + 4-bit taban. Sadece ~%0.5 parametre eğitilir.
> **Analoji:** Koca bir kütüphaneyi (donuk model) baştan yazmak yerine, kenara birkaç **not kağıdı** (LoRA adaptörü) iliştiririz. Kütüphane aynı kalır; notlar görevi öğrenir.


## 1. Kurulum

**Ne:** Unsloth ve bağımlılıklarını kuruyoruz. **Niçin koşullu?** Colab ile Kaggle farklı önyüklü ortamlara sahip; yanlış paket sürümü çakışma yaratır. Aşağısı Unsloth'un resmi, iki ortamda da çalışan kalıbıdır.

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    # Kaggle / yerel
    !pip install unsloth
else:
    # Colab: önyüklü torch'u bozmamak için bağımlılıkları sabitle
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

In [ ]:
# GPU teyidi — hangi kartla, ne kadar bellekle çalışıyoruz?
import torch, unsloth
print("Unsloth :", unsloth.__version__)
print("GPU     :", torch.cuda.get_device_name(0))
print("VRAM    :", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

## 2. Modeli 4-bit yükle

**Ne:** `unsloth/gemma-4-E4B-it` modelini 4-bit olarak indiriyoruz.
**Niçin bu model?** `-it` = instruct (komut anlar). `E4B` = *efektif* ~4B aktif parametre; toplam ~8B ama Gemma 4 mimarisi her adımda bunun bir kısmını çalıştırır → T4'e sığar.
**Niçin `load_in_4bit=True`?** Yukarıda anlattığımız nicemleme: belleğe sığdırmak için.
**Niçin `max_seq_length=2048`?** Bir örnekte *şema + metin + hedef JSON* birlikte bu token bütçesine sığmalı. Büyütmek eğitim süresini ve belleği artırır — gereksizse büyütmeyin.

> "gated model" hatası alırsanız: Kaggle → *Add-ons → Secrets* → `HF_TOKEN` ekleyip `from huggingface_hub import login; login(os.environ["HF_TOKEN"])`.

In [ ]:
from unsloth import FastModel
import torch

MAX_SEQ_LEN = 2048

model, tokenizer = FastModel.from_pretrained(
    model_name      = "unsloth/gemma-4-E4B-it",
    dtype           = None,     # otomatik (T4'te float16)
    max_seq_length  = MAX_SEQ_LEN,
    load_in_4bit    = True,     # QLoRA tabanı: 4-bit
    full_finetuning = False,    # tam fine-tune DEĞİL → adapter eğiteceğiz
)

### LoRA adaptörünü ekle

**Ne:** Donuk modele küçük, eğitilebilir LoRA matrisleri iliştiriyoruz.
**Niçin `finetune_vision_layers=False`?** Gemma 4 çok-kipli (metin+görsel), ama bizim görev sadece metin → görü katmanlarını eğitmek boşa bellek/zaman harcar.
**Niçin `r=8, lora_alpha=8`?** `r` adaptörün kapasitesi. Küçük veri (484 örnek) için 8 yeterli; büyük/zor görevde 16–32 denenir. `alpha ≈ r` önerilir.

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # sadece metin görevi
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r            = 8,
    lora_alpha   = 8,
    lora_dropout = 0,
    bias         = "none",
    random_state = 3407,
)
model.print_trainable_parameters()  # eğitilen / toplam parametre → PEFT'in gücünü görün

## 3. Veriyi yükle ve tanı

**Ne:** `paraloq/json_data_extraction` — her satırda **`text`** (dağınık belge), **`schema`** (hedef JSON şeması) ve **`item`** (doğru JSON) var; üçü de string.
**Niçin hazır veri?** Bugün yöntemi öğreniyoruz; hazır, temiz bir setle hattın tamamını görüyoruz.
**Gerçek hayatta:** Bir firmada/kamuda çalışırken HF'deki tertemiz set işinize yaramaz — *kendi* verinizi hazırlamanız gerekir. (Ödevlerde bunu yapacağız.)

In [ ]:
from datasets import load_dataset

raw = load_dataset("paraloq/json_data_extraction", split="train")
print(raw)
print("\nAlanlar:", raw.column_names)
print("Konular:", sorted(set(raw["topic"])))

ornek = raw[0]
print("\n--- METİN (ilk 400 karakter) ---\n", ornek["text"][:400])
print("\n--- HEDEF JSON (item) ---\n", ornek["item"][:400])

### Uzunluk filtresi — niçin?

Bazı belgeler çok uzun (127K karaktere kadar) ve 2048 token'a **sığmaz**; model onları göremez, eğitim de bozulur.
**Kural:** Girdi context window'a sığmıyorsa işe yaramaz. Kaba bir karakter eşiğiyle uzun örnekleri ayıklıyoruz.

In [ ]:
import json

def toplam_uzunluk(row):
    return len(row["schema"]) + len(row["text"]) + len(row["item"])

filt = raw.filter(lambda r: toplam_uzunluk(r) < 4500)
print(f"Filtre sonrası: {len(filt)} / {len(raw)} örnek kaldı")

## 4. Sohbet formatına çevir

**Ne:** Her örneği bir *konuşmaya* dönüştürüyoruz — **user**: talimat + şema + metin; **assistant**: sadece hedef JSON.
**Niçin?** Instruct model, eğitim verisini de sohbet formatında bekler. Modele "girdi buysa, çıktın *tam olarak* bu JSON olsun" diye örnek gösteriyoruz.

**Gemma 4 özel noktalar (niçin böyle):**
- `get_chat_template(..., "gemma-4")`: Gemma 4'ün özel `<start_of_turn>` etiketlerini uygular. **"gemma-4"** (thinking değil) seçiyoruz — düşünme modu JSON çıktısını kirletir.
- İçerik `[{"type":"text","text": ...}]` liste formatında: Gemma 4 çok-kipli olduğu için mesaj içeriği liste bekler.
- `.removeprefix('<bos>')`: şablon başa bir `<bos>` ekler; eğitici de eklerse **çift bos** olur → onu kırpıyoruz.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template = "gemma-4")

TALIMAT = (
    "Aşağıdaki metinden, verilen JSON şemasına birebir uyan geçerli bir JSON üret. "
    "Yalnızca JSON döndür; açıklama, giriş cümlesi veya kod bloğu işareti ekleme."
)

def sohbete_cevir(row):
    hedef = json.dumps(json.loads(row["item"]), ensure_ascii=False)  # compact, tutarlı hedef
    kullanici = f"{TALIMAT}\n\n### ŞEMA:\n{row['schema']}\n\n### METİN:\n{row['text']}"
    return {"conversations": [
        {"role": "user",      "content": [{"type": "text", "text": kullanici}]},
        {"role": "assistant", "content": [{"type": "text", "text": hedef}]},
    ]}

konusma = filt.map(sohbete_cevir, remove_columns=filt.column_names)

def formatla(batch):
    metinler = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")
        for c in batch["conversations"]
    ]
    return {"text": metinler}

veri = konusma.map(formatla, batched=True, remove_columns=["conversations"])
print(veri[0]["text"][:600])

### Eğitim / test ayrımı — niçin şart?

Modeli **eğitimde hiç görmediği** örneklerde değerlendireceğiz. Eğitim verisinde test edersek model "ezberi" yüzünden yanıltıcı yüksek skor verir (**data leakage**). O yüzden şimdi ayırıyoruz.

In [ ]:
bolunmus = veri.train_test_split(test_size=0.1, seed=3407)
train_ds, test_ds = bolunmus["train"], bolunmus["test"]
print("Eğitim:", len(train_ds), " | Test:", len(test_ds))

# Baseline karşılaştırması için ham (formatlanmamış) test örnekleri:
ham_test = filt.train_test_split(test_size=0.1, seed=3407)["test"]

## 5. ⚖️ Baseline — fine-tune ÖNCESİ ne yapıyor?

**Niçin önce ölçüyoruz?** Bilimsel disiplin: iyileşmeyi kanıtlamak için başlangıç noktasını kaydetmeliyiz. "Sonra iyi oldu" demek yetmez; **öncesini** bilmeden iyileşme iddiası havada kalır.
**Metrik:** Çıktı `json.loads()` ile parse ediliyor mu? (geçerli-JSON oranı) — objektif, tartışmasız.

In [ ]:
from transformers import TextStreamer

def istem(row):
    kullanici = f"{TALIMAT}\n\n### ŞEMA:\n{row['schema']}\n\n### METİN:\n{row['text']}"
    return [{"role": "user", "content": [{"type": "text", "text": kullanici}]}]

def cikar(messages, max_new_tokens=512):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_tensors="pt", return_dict=True,
    ).to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         temperature=1.0, top_p=0.95, top_k=64)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def gecerli_json_orani(orneklerden):
    basari = 0
    for row in orneklerden:
        try:
            json.loads(cikar(istem(row)).strip()); basari += 1
        except Exception:
            pass
    return basari / len(orneklerden)

degerlendirme = list(ham_test)[:20]   # hız için 20 örnek
baseline_oran = gecerli_json_orani(degerlendirme)
print(f"\n📉 BASELINE geçerli-JSON oranı: {baseline_oran:.0%}")

Bir örnekte çıktıya gözle bakalım. Fine-tune öncesi model çoğu zaman doğruya *yakın* üretir ama **gevezelik / kod bloğu / şemadan sapma** yapar — işte fine-tune'un çözeceği sorun:

In [ ]:
print(cikar(istem(degerlendirme[0]))[:800])

## 6. Eğitim (SFT — Supervised Fine-Tuning)

**Ne:** `SFTTrainer` ile denetimli ince ayar. Modele "bu girdiye bu çıktı" örneklerini gösterip LoRA adaptörünü güncelliyoruz.
**Niçin `max_steps=60`?** Bu bir **demo** (~10–15 dk). Gerçek projede `max_steps`'i kaldırıp `num_train_epochs=1..3` kullanın.
**Niçin `batch_size=1` + `gradient_accumulation_steps=4`?** E4B görece büyük; T4'te tek örnek sığar. Gradyan biriktirme ile *sanki* 4'lük batch eğitmiş gibi oluruz — bellek şişmeden.
**Niçin `train_on_responses_only`?** Model sadece **cevap** (JSON) tokenlarından öğrensin; uzun şema+metin girişini kayıptan dışlarız → daha verimli, girdiyi ezberlemez.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = None,
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,   # efektif batch = 4
        warmup_steps                = 5,
        max_steps                   = 60,  # DEMO. Gerçekte: num_train_epochs=1..3
        learning_rate               = 2e-4,
        logging_steps               = 1,
        optim                       = "adamw_8bit",
        weight_decay                = 0.001,
        lr_scheduler_type           = "linear",
        seed                        = 3407,
        report_to                   = "none",
    ),
)

# Sadece 'model' (cevap) turlarından öğren — marker'ları Unsloth otomatik bulur:
trainer = train_on_responses_only(trainer)

In [ ]:
istatistik = trainer.train()
print("\nSon eğitim kaybı (loss):", round(istatistik.training_loss, 4))
# Loss düşüyorsa: model JSON kalıbını öğreniyor demektir.

## 7. 📈 Fine-tune SONRASI — aynı testte ölç

**Niçin aynı 20 örnek?** Elmayla elmayı kıyaslamak için. Tek değişen: model artık eğitildi. Farkı **bu** izole eder.

In [ ]:
finetuned_oran = gecerli_json_orani(degerlendirme)
print(f"📉 Baseline    : {baseline_oran:.0%}")
print(f"📈 Fine-tuned  : {finetuned_oran:.0%}")
print(f"Δ İyileşme     : {(finetuned_oran - baseline_oran):+.0%}")

Aynı örnekte çıktı — artık temiz, şemaya uygun, doğrudan parse edilebilir JSON olmalı:

In [ ]:
cikti = cikar(istem(degerlendirme[0]))
print(cikti[:800])
print("\n--- json.loads testi ---")
try:
    parsed = json.loads(cikti.strip())
    print("✅ Geçerli JSON. Üst düzey anahtarlar:", list(parsed.keys()))
except Exception as e:
    print("❌ Parse hatası:", e)

## 8. Modeli kaydet

**Ne:** Sadece LoRA adaptörünü kaydediyoruz (~50–100 MB). **Niçin sadece adaptör?** Taban model zaten Hugging Face'te; adaptör = bizim öğrettiğimiz "fark". Küçük, paylaşması kolay.

In [ ]:
model.save_pretrained("gemma4-json-lora")
tokenizer.save_pretrained("gemma4-json-lora")
print("✅ Adaptör kaydedildi: gemma4-json-lora/")
# Kaggle'da bu klasör Output olarak indirilir / HF'ye Model olarak yüklenebilir.

### (Bonus) Ollama / llama.cpp için GGUF'a çıkar

**Niçin GGUF?** Yerelde (laptop, sunucu) llama.cpp/Ollama ile çalıştırmanın standart formatı — API'siz, offline, ücretsiz çıkarım. Taban + adaptör birleştirilip tek dosya olur. Zaman/yer ister; sınıfta opsiyonel.

In [ ]:
# model.save_pretrained_gguf("gemma4-json-gguf", tokenizer, quantization_method="q4_k_m")
# Sonra: ollama create json-cikar -f Modelfile   (FROM ./gemma4-json-gguf/...)
print("GGUF export için üstteki satırı açın (birkaç dakika sürer).")

## 9. Özet — neyi niçin yaptık?

| Adım | Ne | Niçin |
|---|---|---|
| Model 4-bit | Gemma 4 E4B'yi 4-bit yükledik | Ücretsiz T4'e sığsın |
| LoRA | Küçük adaptör ekledik | Tam fine-tune pahalı; ~%0.5 parametre yeter |
| Sohbet formatı | user=şema+metin, assistant=JSON | Modele hedef davranışı örnekle göster |
| train/test | Test setini ayırdık | Data leakage'ı önle, dürüst ölç |
| Baseline | Öncesini ölçtük | İyileşmeyi kanıtlamak için başlangıç |
| SFT | Adaptörü eğittik | JSON disiplinini ağırlıklara işle |
| Sonrası | Aynı testte ölçtük | Farkı objektif göster (json.loads oranı) |

**Ödevler / genişletme:**
- `max_steps` yerine `num_train_epochs=2` ile tam eğitim; oranı tekrar ölçün.
- Değerlendirmeyi güçlendirin: `jsonschema.validate` ile **şemaya uygunluk** + alan **F1**.
- `r` / `lora_alpha` (8→16→32) etkisini gözlemleyin.
- **Kendi Türkçe verinizi** hazırlayın (fatura/dilekçe → JSON) ve aynı hattı çalıştırın — gerçek dünyada asıl beceri budur.
- GGUF + Ollama ile yerel bir mini çıkarım API'si kurun.

---
*Antalya LLM Tabanlı Uygulama Geliştirme Eğitimi — Dr. Murat Altun*
